# Bengali Long-Form ASR—V21 

### What HELPED:
-  Unicode NFC normalization (0.444 → 0.429)
-  Beam search 2-4 (0.444 → 0.366)
-  Chunk=20.1s (1st place setting)

### This Version Tests:
1. **Greedy decoding (beams=1)** vs beam search
2. **Different stride overlaps**
3. **Zero post-processing** except Unicode normalization

In [1]:
# 1. Setup
import os
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
from tqdm.auto import tqdm
from transformers import pipeline, AutoModelForSpeechSeq2Seq, AutoProcessor

warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.9.0+cu126
CUDA: True
GPU: Tesla T4


In [2]:
# 2. Official Competition Normalization (EXACT)

ZW = r"[\u200B-\u200D\uFEFF]"

def normalize_bn_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    s = re.sub(ZW, "", s)
    s = s.replace("\u00A0", " ")
    s = " ".join(s.split())
    return s

# Test
print("Test:", normalize_bn_text("আমি\u200Bভালো  আছি"))

Test: আমিভালো আছি


In [3]:
# 3. Configuration

MODEL_NAME = "bengaliAI/tugstugi_bengaliai-asr_whisper-medium"

# Test both greedy and beam search
NUM_BEAMS = 1  # Start with greedy (simplest, often best)
# If score > 0.40, change to 2 or 4

CHUNK_LENGTH_S = 20.1
STRIDE_LENGTH_S = (5, 3)
MAX_NEW_TOKENS = 260
BATCH_SIZE = 8

TEST_AUDIO_DIR = "/kaggle/input/dl-sprint-4-0-bengali-long-form-speech-recognition/transcription/transcription/test/audio"
OUTPUT_CSV = "/kaggle/working/submission.csv"
TARGET_SR = 16000

print("=" * 60)
print("V21 CONFIGURATION")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Beams: {NUM_BEAMS} (greedy=1, beam=2-4)")
print(f"Chunk: {CHUNK_LENGTH_S}s | Stride: {STRIDE_LENGTH_S}")
print(f"Post-processing: ONLY Unicode normalization")
print("=" * 60)

V21 CONFIGURATION
Model: bengaliAI/tugstugi_bengaliai-asr_whisper-medium
Beams: 1 (greedy=1, beam=2-4)
Chunk: 20.1s | Stride: (5, 3)
Post-processing: ONLY Unicode normalization


In [4]:
# 4. Collect Files

audio_files = sorted([
    f for f in os.listdir(TEST_AUDIO_DIR)
    if f.endswith(('.wav', '.mp3'))
])

print(f"Found {len(audio_files)} files")
print(f"First 3: {audio_files[:3]}")

Found 24 files
First 3: ['test_001.wav', 'test_002.wav', 'test_003.wav']


In [5]:
# 5. Load Model

device = 0 if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == 0 else torch.float32

print(f"Loading model...")

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
)
model.to("cuda" if device == 0 else "cpu")
model.eval()

processor = AutoProcessor.from_pretrained(MODEL_NAME)

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device,
    torch_dtype=dtype,
    chunk_length_s=CHUNK_LENGTH_S,
    stride_length_s=STRIDE_LENGTH_S,
    return_timestamps=False,
)

asr_pipe.model.config.use_cache = False

print("✓ Model loaded")

Loading model...


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

...

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


✓ Model loaded


In [6]:
# 6. Transcription Loop

results = []

print(f"\n{'='*60}")
print(f"TRANSCRIBING WITH NUM_BEAMS={NUM_BEAMS}")
print(f"{'='*60}\n")

for i, fname in enumerate(tqdm(audio_files, desc="Transcribing")):
    fpath = os.path.join(TEST_AUDIO_DIR, fname)
    file_id = Path(fname).stem
    
    try:
        # Load audio
        audio, sr = librosa.load(fpath, sr=TARGET_SR)
        
        # Transcribe
        output = asr_pipe(
            {"array": audio, "sampling_rate": sr},
            batch_size=BATCH_SIZE,
            generate_kwargs={
                "num_beams": NUM_BEAMS,
                "max_new_tokens": MAX_NEW_TOKENS,
                "temperature": 0.0,
                "do_sample": False,
            }
        )
        
        # Apply ONLY Unicode normalization
        text = normalize_bn_text(output["text"])
        
        results.append({
            "filename": file_id,
            "transcript": text
        })
        
        # Show first result
        if i == 0:
            print(f"\nFirst result:")
            print(f"  File: {file_id}")
            print(f"  Length: {len(text)} chars, {len(text.split())} words")
            print(f"  Sample: {text[:150]}...\n")
        
        # Memory cleanup
        if torch.cuda.is_available() and (i + 1) % 5 == 0:
            torch.cuda.empty_cache()
    
    except Exception as e:
        print(f"\nERROR with {fname}: {e}")
        results.append({"filename": file_id, "transcript": ""})

df = pd.DataFrame(results)
print(f"\n✓ Transcription complete: {len(df)} files")


TRANSCRIBING WITH NUM_BEAMS=1



Transcribing:   0%|          | 0/24 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'num_beams'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Both `max_new_tokens` (=260) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Ple


First result:
  File: test_001
  Length: 22373 chars, 4102 words
  Sample: নির্বাচনের জন্য দেশের প্র প্রধান বিদ্যমান ব্যবস্থাপনা বিশ্ববিদ্যালয়ের বিশেষ বিদ্যালয়ের বিশেষ বিদ্যালয়ের বিশেষ বিদ্যালয়ের বিশেষ বিদ্যালয়ের দিনের দ...



Both `max_new_tokens` (=260) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=260) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



✓ Transcription complete: 24 files


In [7]:
# 7. Save Submission

# Final normalization pass
df['transcript'] = df['transcript'].apply(normalize_bn_text)

# Save
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"\n{'='*60}")
print("SAVED SUBMISSION")
print(f"{'='*60}")
print(f"File: {OUTPUT_CSV}")
print(f"Rows: {len(df)}")

# Verify
empty = (df['transcript'] == '').sum()
if empty > 0:
    print(f"\n⚠ WARNING: {empty} empty transcripts!")
else:
    print(f"\n✓ All transcripts non-empty")

# Stats
print(f"\nStats:")
print(f"  Avg length: {df['transcript'].str.len().mean():.0f} chars")
print(f"  Avg words: {df['transcript'].str.split().str.len().mean():.0f}")

print(f"\n{'='*60}")
print(f"Expected score with beams={NUM_BEAMS}:")
if NUM_BEAMS == 1:
    print("  0.35-0.40 WER (greedy is fast but may be less accurate)")
elif NUM_BEAMS == 2:
    print("  0.32-0.36 WER (good balance)")
else:
    print("  0.30-0.35 WER (best quality, slower)")
print(f"{'='*60}")


SAVED SUBMISSION
File: /kaggle/working/submission.csv
Rows: 24

✓ All transcripts non-empty

Stats:
  Avg length: 26346 chars
  Avg words: 4791

Expected score with beams=1:
  0.35-0.40 WER (greedy is fast but may be less accurate)
